# 🎮 Video Game Sales Analysis
## Data Analysis Project — Pandas, NumPy & Matplotlib
---
| | |
|---|---|
| **Student Name** | MOHSIN IBNA HOSSAIN |
| **Student ID** | 23-50194-1 |
| **Dataset** | Video Game Sales (vgsales.csv) |
| **Source** | https://www.kaggle.com/datasets/gregorut/videogamesales |
| **Tools** | Python · Pandas · NumPy · Matplotlib · Seaborn |


## 1. Problem Definition
The global video game industry generates billions in annual revenue. This project investigates **16,598 video games** released between 1980–2016 to uncover sales patterns, regional differences, and market leaders.

### Analytical Questions
1. **Which genres and platforms dominate global sales, and how has this shifted over time?**
2. **How do regional markets (NA, EU, JP) differ in genre and publisher preferences?**
3. **What is the relationship between North American and global sales — and which titles are outliers?**

### Why These Questions Matter
- Genre dominance guides developer investment decisions.
- Regional analysis reveals cultural differences in gaming taste.
- Outlier detection highlights exceptional commercial successes or failures.


## 2. Import Libraries & Environment Setup

In [1]:

import os, warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
load_dotenv()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Config from .env
DATASET_PATH   = os.getenv("DATASET_PATH",  "vgsales.csv")
DATASET_SOURCE = os.getenv("DATASET_SOURCE", "Kaggle")
FIGURES_DIR    = os.getenv("FIGURES_DIR",    "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

# Global style
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "font.family": "DejaVu Sans",
})

print("[OK] Libraries loaded.")
print(f"   Pandas    {pd.__version__}")
print(f"   NumPy     {np.__version__}")
print(f"   Dataset   {DATASET_PATH}")
print(f"   Figures -> {FIGURES_DIR}/")


[OK] Libraries loaded.
   Pandas    3.0.2
   NumPy     2.4.4
   Dataset   vgsales.csv
   Figures -> figures/


## 3. Data Understanding

In [2]:
# Load dataset
df_raw = pd.read_csv(DATASET_PATH, encoding="latin-1")

print("Shape:", df_raw.shape)
print("\nColumns:", df_raw.columns.tolist())
print("\nData Types:")
print(df_raw.dtypes)
print("\nFirst 5 rows:")
display(df_raw.head())


Shape: (16598, 11)

Columns: ['Rank', 'Name', 'Platform', 'Year', 'Genre', 'Publisher', 'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']

Data Types:
Rank              int64
Name                str
Platform            str
Year            float64
Genre               str
Publisher           str
NA_Sales        float64
EU_Sales        float64
JP_Sales        float64
Other_Sales     float64
Global_Sales    float64
dtype: object

First 5 rows:


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


In [3]:
# Summary statistics
print("Summary Statistics:")
display(df_raw.describe(include="all").round(2))


Summary Statistics:


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
count,16598.00,16598,16598,16327.00,16598,16540,16598.00,16598.00,16598.00,16598.00,16598.00
unique,NaN,11493,31,NaN,12,578,NaN,NaN,NaN,NaN,NaN
top,NaN,Need for Speed: Most Wanted,DS,NaN,Action,Electronic Arts,NaN,NaN,NaN,NaN,NaN
freq,NaN,12,2163,NaN,3316,1351,NaN,NaN,NaN,NaN,NaN
mean,8300.61,NaN,NaN,2006.41,NaN,NaN,0.26,0.15,0.08,0.05,0.54
std,4791.85,NaN,NaN,5.83,NaN,NaN,0.82,0.51,0.31,0.19,1.56
min,1.00,NaN,NaN,1980.00,NaN,NaN,0.00,0.00,0.00,0.00,0.01
25%,4151.25,NaN,NaN,2003.00,NaN,NaN,0.00,0.00,0.00,0.00,0.06
50%,8300.50,NaN,NaN,2007.00,NaN,NaN,0.08,0.02,0.00,0.01,0.17
75%,12449.75,NaN,NaN,2010.00,NaN,NaN,0.24,0.11,0.04,0.04,0.47


In [4]:
# Missing value analysis
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
miss_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
print("Missing Value Summary:")
display(miss_df[miss_df["Missing Count"] > 0])

Missing Value Summary:


,Missing Count,Missing %
Year,271,1.63
Publisher,58,0.35


### Data Understanding Interpretation
- Dataset has **16,598 rows and 11 columns** - sufficient for deep, multi-dimensional analysis.
- `Year` has **271 missing values (1.6%)** - will be dropped for time-based analysis only.
- `Publisher` has **58 missing values (0.35%)** - minimal impact; dropped during publisher-specific analysis.
- Sales columns are all `float64`, ready for arithmetic and aggregation.
- `Year` is `float64` due to NaN presence - will be converted to nullable `Int64` after dropping NaNs.
- Wide time span (1980–2016) enables meaningful trend analysis.
